In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os 

In [2]:
import sys
sys.path.append("..")

In [3]:
from utils.iv_solver import iv_newton
from utils.get_market_data import get_candles

In [4]:
from datetime import datetime, timedelta
now = datetime.now()
from_ = now - timedelta(days=1)
sber_price = get_candles("SBER", from_, now)

2026-05-21 07:46:06.620569
Number of deleted duplicates: 0


In [18]:
sber_price.head(1)
curr_sber_price = sber_price.iloc[-1]['close']
print(curr_sber_price)

325.68


In [6]:
import numpy as np
from itertools import product

call_suff = ["CE6", "CE6D", "CF6A"]
put_suff = ["CQ6", "CQ6D", "CR6A"]
expiry_dates = [datetime(2026, 5, 20), datetime(2026, 5, 27), datetime(26, 6, 3)]
strikes = np.linspace(270, 380, 12, dtype=int)

put_tickers  = [f"SR{s}{suf}" for s, suf in product(strikes, put_suff)]
call_tickers = [f"SR{s}{suf}" for s, suf in product(strikes, call_suff)]
len(call_tickers)

36

In [7]:
load_dotenv()
db_url = os.getenv("DB_URL")

In [8]:
import psycopg2

conn = psycopg2.connect(db_url)
cur = conn.cursor()

In [10]:
rows_arr = []
for ticker in call_tickers:
    cur.execute("""
    SELECT ticker, bids, asks FROM orderbooks
    WHERE ticker = %s
    ORDER BY timestamp ASC LIMIT 1
    """, (ticker,))
    rows = cur.fetchall()
    rows_arr.extend(rows)

In [21]:
df = pd.DataFrame(rows_arr, columns = ["ticker", "bids", "asks"])
df['best_bid'] = df['bids'].apply(lambda x: x[0]['price'] if x else None)
df['best_ask'] = df['asks'].apply(lambda x: x[0]['price'] if x else None)

df['mid'] = ((df['best_ask'] + df['best_bid']) / 2).fillna(df['best_ask']).fillna(df['best_bid'])

In [22]:
df = df[["ticker", "mid"]]
df

,ticker,mid
0,SR270CE6,55.080
1,SR270CE6D,51.935
2,SR270CF6A,55.240
3,SR280CE6,45.400
4,SR280CE6D,42.085
5,SR280CF6A,48.240
6,SR290CE6,35.230
7,SR290CE6D,36.660
8,SR290CF6A,39.070
9,SR300CE6,0.020
